# Basis Transforms

Transform vector and tensor components between curvilinear and Cartesian bases and check the round trip.

Navigation: [Index](../index.ipynb) | Previous: [Reference-Metric Applications](reference_metric_applications.ipynb) | Next: [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb)


## Learning Goals

- Transform vector and tensor components between bases.
- Separate the physical vector from the numbers used to describe it.
- Use round-trip residuals to check that a transform can be undone.

## Words for This Notebook

- **Basis:** the local directions used to describe components of a vector or tensor.
- **Component:** one number in a vector or tensor description.
- **Round trip:** transforming to another basis and then back again.
- **Residual:** the leftover after subtracting the original expression.

Use the code cells actively: first predict what should happen, then run the cell, then explain the output in plain language. This predict-run-explain pattern keeps the physics idea connected to the programming details.


## Transform and Check Components
A radial vector in the reference-metric basis becomes a Cartesian vector with direction-dependent components. Round-trip residuals should vanish.

## Import SymPy for Component Checks

These imports expose the NRPy and Python tools used in the next steps.


In [1]:
import sympy as sp


## Import Basis Transform Helpers

These imports expose the NRPy registries and infrastructure writers used below.


In [2]:
import nrpy.indexedexp as ixp
import nrpy.reference_metric as refmetric
from nrpy.equations.basis_transforms.jacobians import BasisTransforms


## Build the Spherical Reference Metric

The output connects coordinate maps to metric factors.


In [3]:
spherical = refmetric.reference_metric["Spherical"]
transforms = BasisTransforms("Spherical")
print("reference metric:", spherical.CoordSystem)
print("scale factors:", spherical.scalefactor_orthog)


Setting up reference_metric[Spherical]...


reference metric: Spherical
scale factors: [1, xx0, xx0*sin(xx1)]


## Step 4: Transform a radial vector

The output rewrites a reference-metric-basis vector in Cartesian components.

In [4]:
radial_amplitude = sp.Symbol("V_r", real=True)
radial_vectorU = [radial_amplitude, sp.sympify(0), sp.sympify(0)]
cartesian_vectorU = transforms.basis_transform_vectorU_from_rfmbasis_to_Cartesian(
    radial_vectorU
)
print("Cartesian components of a radial vector:")
for component in cartesian_vectorU:
    print(sp.trigsimp(component))


Cartesian components of a radial vector:
V_r*sin(xx1)*cos(xx2)
V_r*sin(xx1)*sin(xx2)
V_r*cos(xx1)


## Verify a Vector Round Trip

A zero residual confirms that the symbolic expression matches the expected identity.


In [5]:
generic_vectorU = ixp.declarerank1("V")
cartesian_genericU = transforms.basis_transform_vectorU_from_rfmbasis_to_Cartesian(
    generic_vectorU
)
round_trip_vectorU = transforms.basis_transform_vectorU_from_Cartesian_to_rfmbasis(
    cartesian_genericU
)
round_trip_residuals = [
    sp.trigsimp(sp.simplify(round_trip_vectorU[i] - generic_vectorU[i]))
    for i in range(3)
]
print("contravariant vector residuals:", round_trip_residuals)
if round_trip_residuals != [0, 0, 0]:
    raise RuntimeError("Expected vector round-trip residuals to vanish.")


contravariant vector residuals: [0, 0, 0]


## Verify the Cartesian Metric Diagonal

A zero residual confirms that the symbolic expression matches the expected identity.


In [6]:
cartesian_metricDD = transforms.basis_transform_tensorDD_from_rfmbasis_to_Cartesian(
    transforms.rfm.ghatDD
)
diagonal_residuals = [
    sp.trigsimp(sp.simplify(cartesian_metricDD[i][i] - 1)) for i in range(3)
]
off_diagonal_residual = sp.trigsimp(sp.simplify(cartesian_metricDD[0][1]))
print("Cartesian metric diagonal residuals:", diagonal_residuals)
print("Cartesian metric (0, 1) residual:", off_diagonal_residual)
if diagonal_residuals != [0, 0, 0] or off_diagonal_residual != 0:
    raise RuntimeError("Expected transformed metric residuals to vanish.")


Cartesian metric diagonal residuals: [0, 0, 0]
Cartesian metric (0, 1) residual: 0


The transformed components show how vector and tensor data change basis while representing the same geometric object. This is the same bookkeeping used when comparing ADM/BSSN quantities across coordinate choices.


## Learning Check

Before the radial-vector transform, predict whether Cartesian x, y, and z components should all be nonzero away from the axes.


## Continue to the Curvilinear Wave Equation
- [Reference-Metric Applications](reference_metric_applications.ipynb)
- [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb)
- [ADM and BSSN Conversions](../6-numerical_relativity/adm_bssn_conversions.ipynb)
